In [32]:
import os
import re

In [33]:
def parse_routine_file(routine_path):
    
    if not os.path.isfile(routine_path):
        print(f"File not found: {routine_path}")
        return []

    with open(routine_path, 'r', encoding="utf-8") as f:
        routine_content = f.read()

    routines = []
    blocks = []
    current_location = None  # Store the current room location

    for line in routine_content.splitlines():
        line = line.strip()

        # Skip empty lines
        if not line:
            continue
      
        location_match = re.match(r"^(\d{1,2}:\d{1,2}) - \d{1,2}:\d{1,2}, (.+?): .+", line)
        
        if location_match:
            # If we have collected steps, save them before moving to a new block
            if blocks:
                routines.append(blocks)
                blocks = []
            
            start_time = location_match.group(1).strip().lower().replace(" ", "")
            current_location = location_match.group(2).strip().lower().replace(" ", "")
            blocks.append(f"[walk] <{current_location}> ({start_time} - {start_time}) ({current_location})")
            continue  # Skip adding this line to routines

        # Match step pattern "Step X: description"
        step_match = re.match(r"^Step\s+(\d+):\s*(.+)", line)
        if step_match:
            step_num = step_match.group(1)
            step_desc = step_match.group(2)
            blocks.append(f"{step_desc} ({current_location})")  # Append with location

    # Ensure the last block gets added
    if blocks:
        routines.append(blocks)

    return routines

In [34]:
# Clean and crop a single per-day routine file by parsing it into simulator-ready action blocks
def clean_and_crop_daily_routine_file(input_file_path: str, output_dir: str):
    
    if not os.path.isfile(input_file_path):
        print(f"[AgentSense][Error] File not found: {input_file_path.replace(os.sep, '/')}")
        return

    os.makedirs(output_dir, exist_ok=True)

    filename = os.path.basename(input_file_path)
    base_name = os.path.splitext(filename)[0]
    output_path = os.path.join(output_dir, f"{base_name}_parsed.txt")

    print(f"[AgentSense] Cleaning routine file: {filename}")

    parsed_routines = parse_routine_file(input_file_path)

    with open(output_path, "w", encoding="utf-8") as out_file:
        for block in parsed_routines:
            for step in block:
                out_file.write(step + "\n")
            out_file.write("\n")  # Separate routine blocks

    print(f"[AgentSense] Cleaned routine saved to: {output_path.replace(os.sep, '/')}")


In [35]:
# =========================
# Configuration (User-editable)
# =========================

# Single per-day routine file generated in Step 4
INPUT_DAILY_ROUTINE_FILE = "data/step_4_data/Sarah_routine_env_0_Sunday.txt"

# Directory to save cleaned / cropped routine file (Step 5 output)
OUTPUT_CLEAN_ROUTINE_DIR = "data/step_5_data"

# =========================
# Run
# =========================

clean_and_crop_daily_routine_file(
    input_file_path=INPUT_DAILY_ROUTINE_FILE,
    output_dir=OUTPUT_CLEAN_ROUTINE_DIR
)

[AgentSense] Cleaning routine file: Sarah_routine_env_0_Sunday.txt
[AgentSense] Cleaned routine saved to: data/step_5_data/Sarah_routine_env_0_Sunday_parsed.txt
